<a href="https://colab.research.google.com/github/prathameshks/LLM-Finetuning/blob/main/Fine_Tuning-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json

file = json.load(open("training_data.json", "r"))
print(file[1])

{'input': 'Question: How has our membership grown quarter by quarter since 2022?\n\nAvailable database schema:\nTables:\n- dim_person (person_id, first_name, last_name, gender, age_at_join, join_date, state, city, county, zipcode, country, ethnicity, title, email, phone, join_source)\n- fact_membership (membership_id, person_id, product_id, start_date, end_date, is_renewed, is_recurring, churn_reason, dues_paid_thru_date, membership_subscription_id, order_id)\n- dim_product (product_id, product_name, product_type, price, category)\n- dim_order (order_id, person_id, total_amount, order_date, payment_status, channel)\n- dim_membership_status (status_id, status, description)\n- dim_payment (payment_id, order_id, payment_method, amount, payment_date, status)', 'output': {'questionAnswerable': True, 'requiresChart': True, 'requiresTextOutput': False, 'chartType': 'line', 'chartDetails': 'How has our membership grown quarter by quarter since 2022, line chart', 'notAnswerableReason': None}}


In [3]:
# !pip uninstall -y unsloth peft

!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [4]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [5]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
from datasets import Dataset

def format_prompt(example):
    return f"### Input: {example['input']}\n### Output: {json.dumps(example['output'])}<|endoftext|>"

formatted_data = [format_prompt(item) for item in file]
dataset = Dataset.from_dict({"text": formatted_data})

In [7]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.3.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [10]:
# ============================================================
# CLASSIFICATION TASK - Cell 1: Prompt + Eval Utilities + Test Data
# ============================================================
import json, re, torch
from tqdm import tqdm

CLASSIFICATION_PROMPT = (
    "You are a question classifier for a membership analytics system.\n\n"
    "DATABASE SCHEMA:\n"
    "- dim_person: member demographics (person_id, gender, age_at_join, join_date, state, city, county, ethnicity, join_source)\n"
    "- fact_membership: membership records (membership_id, person_id, product_id, start_date, end_date, is_renewed, is_recurring, churn_reason)\n"
    "- dim_product: products/plans (product_id, product_name, product_type, price, category)\n"
    "- dim_order: orders (order_id, person_id, total_amount, order_date, payment_status, channel)\n"
    "- dim_membership_status: status lookup (status_id, status, description)\n"
    "- dim_payment: payments (payment_id, order_id, payment_method, amount, payment_date, status)\n\n"
    "RULES:\n\n"
    "ANSWERABILITY: If the question asks for data not in the schema set questionAnswerable=false.\n\n"
    "SINGLE VALUE (requiresChart=false, requiresTextOutput=true):\n"
    "  how many, what is the total/average/rate, most common X, highest/lowest Y, which segment, count of\n\n"
    "MULTIPLE VALUES (requiresChart=true, requiresTextOutput=false):\n"
    "  by county/age/gender/segment/type, breakdown, distribution, trend, over time,\n"
    "  last N months, compare X vs Y, YoY, month over month\n\n"
    "CHART TYPE (only when requiresChart=true):\n"
    "  line      - time trends: over time, monthly, last N months, YoY, quarterly trend\n"
    "  bar       - category comparisons: by segment/county/type at a single point in time\n"
    "  pie       - percentage/share of total (8 or fewer categories)\n"
    "  area      - cumulative values over time\n"
    "  waterfall - gains vs losses: new vs churned, gained vs lost, net change\n\n"
    "Return ONLY a JSON object, no explanation:\n"
    "{\n"
    "  'questionAnswerable': true or false,\n"
    "  'requiresChart': true or false,\n"
    "  'requiresTextOutput': true or false,\n"
    "  'chartType': 'line' or 'bar' or 'pie' or 'area' or 'waterfall' or null,\n"
    "  'chartDetails': 'brief chart description' or null,\n"
    "  'notAnswerableReason': 'reason' or null\n"
    "}"
)

with open("test_data.json", "r") as f:
    test_data = json.load(f)
print(f"Loaded {len(test_data)} test examples")

def extract_json(text):
    text = re.sub(r'```(?:json)?', '', text).strip()
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        fixed = match.group(0).replace('True', 'true').replace('False', 'false').replace("'", '"')
        try:
            return json.loads(fixed)
        except:
            return None

EVAL_FIELDS = ["questionAnswerable", "requiresChart", "requiresTextOutput", "chartType"]

def evaluate(predictions, ground_truth, label="Model"):
    metrics = {f: {"correct": 0, "total": len(predictions)} for f in EVAL_FIELDS}
    parse_failures = 0
    for pred, gt in zip(predictions, ground_truth):
        expected = gt["output"]
        if pred is None:
            parse_failures += 1
            continue
        for f in EVAL_FIELDS:
            if pred.get(f) == expected.get(f):
                metrics[f]["correct"] += 1
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  {'Field':<28} {'Accuracy':>10}  {'Correct/Total':>14}")
    print(f"  {'-'*52}")
    for f, v in metrics.items():
        acc = v["correct"] / v["total"] * 100
        print(f"  {f:<28} {acc:>9.1f}%  {v['correct']:>6}/{v['total']}")
    overall = sum(v["correct"] for v in metrics.values()) / (len(EVAL_FIELDS) * len(predictions)) * 100
    print(f"  {'-'*52}")
    print(f"  {'Overall accuracy':<28} {overall:>9.1f}%")
    print(f"  Parse failures: {parse_failures}/{len(predictions)}")
    print(f"{'='*55}")
    return metrics

print("Setup complete.")

Loaded 65 test examples
Setup complete.


In [14]:
# ============================================================
# CLASSIFICATION TASK - Cell 2: Test Original Model (Baseline)
# Loads via FastLanguageModel (sets apply_qkv) but uses
# use_cache=False to skip the broken Unsloth fast-decode path.
# ============================================================
from unsloth import FastLanguageModel as FLM_baseline

baseline_model, baseline_tokenizer = FLM_baseline.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("Baseline model loaded.")

def run_baseline(mdl, tok, data):
    preds = []
    mdl.eval()
    for item in tqdm(data, desc="Baseline (original model)"):
        user_msg = CLASSIFICATION_PROMPT + """

Classify this question:
""" + item["input"]
        messages = [{"role": "user", "content": user_msg}]
        tokenized = tok.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        )
        # apply_chat_template returns Tensor or BatchEncoding depending on transformers version
        if isinstance(tokenized, torch.Tensor):
            input_ids = tokenized.to("cuda")
        else:
            input_ids = tokenized["input_ids"].to("cuda")
        attention_mask = torch.ones_like(input_ids)
        with torch.no_grad():
            outputs = mdl.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=80,
                do_sample=False,       # greedy - avoids _sample path
                use_cache=False,       # skips broken Unsloth fast-decode KV path
                pad_token_id=tok.eos_token_id,
            )
        new_tokens = outputs[0][input_ids.shape[1]:]
        preds.append(extract_json(tok.decode(new_tokens, skip_special_tokens=True)))
    return preds

baseline_preds = run_baseline(baseline_model, baseline_tokenizer, test_data)
baseline_metrics = evaluate(baseline_preds, test_data, label="ORIGINAL MODEL (Baseline)")

print("--- Sample predictions (original model) ---")
for i in [0, 1, 5]:
    print(f"Q: {test_data[i]['input'][:80]}...")
    print(f"  Expected : {test_data[i]['output']}")
    print(f"  Predicted: {baseline_preds[i]}")

import gc
del baseline_model
torch.cuda.empty_cache()
gc.collect()
print("Baseline model freed from GPU memory.")

==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


Baseline model loaded.


Baseline (original model):   0%|          | 0/65 [00:00<?, ?it/s]Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.w


  ORIGINAL MODEL (Baseline)
  Field                          Accuracy   Correct/Total
  ----------------------------------------------------
  questionAnswerable                64.6%      42/65
  requiresChart                     69.2%      45/65
  requiresTextOutput                38.5%      25/65
  chartType                         64.6%      42/65
  ----------------------------------------------------
  Overall accuracy                  59.2%
  Parse failures: 9/65
--- Sample predictions (original model) ---
Q: Question: How many members are currently active?

Available database schema:
Tab...
  Expected : {'questionAnswerable': True, 'requiresChart': False, 'requiresTextOutput': True, 'chartType': None, 'chartDetails': None, 'notAnswerableReason': None}
  Predicted: {'questionAnswerable': True, 'requiresChart': False, 'requiresTextOutput': True, 'chartType': None, 'chartDetails': None, 'notAnswerableReason': None}
Q: Question: Show me new members gained vs churned each month for t

In [19]:
print("--- Sample predictions (original model) ---")
for i in range(65):
    print(f"\n\n{i+1}, Q: {test_data[i]['input'][:50]}...")
    print(f"  Expected : {test_data[i]['output']}")
    print(f"  Predicted: {baseline_preds[i]}")

--- Sample predictions (original model) ---


1, Q: Question: How many members are currently active?

...
  Expected : {'questionAnswerable': True, 'requiresChart': False, 'requiresTextOutput': True, 'chartType': None, 'chartDetails': None, 'notAnswerableReason': None}
  Predicted: {'questionAnswerable': True, 'requiresChart': False, 'requiresTextOutput': True, 'chartType': None, 'chartDetails': None, 'notAnswerableReason': None}


2, Q: Question: Show me new members gained vs churned ea...
  Expected : {'questionAnswerable': True, 'requiresChart': True, 'requiresTextOutput': False, 'chartType': 'waterfall', 'chartDetails': 'Waterfall chart showing monthly member gains (new joins) vs losses (churn) over last 12 months - green bars for gained, red bars for lost', 'notAnswerableReason': None}
  Predicted: {'questionAnswerable': True, 'requiresChart': True, 'requiresTextOutput': False, 'chartType': 'waterfall', 'chartDetails': 'gains vs losses: new vs churned, gained vs lost, net change',

In [15]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/418 [00:00<?, ? examples/s]

In [20]:
# Train the model
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 418 | Num Epochs = 3 | Total steps = 159
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss
25,0.603585
50,0.106950
75,0.084254
100,0.084520
125,0.064149
150,0.056765


In [22]:
# ============================================================
# CLASSIFICATION TASK - Cell 4: Test Fine-tuned Model + Compare
# Saves before/after results to classification_results.txt
# ============================================================

def run_finetuned(mdl, tok, data):
    preds = []
    mdl.eval()
    for item in tqdm(data, desc="Fine-tuned model"):
        prompt = "### Input: " + item["input"] + "\n### Output: "
        inputs = tok(prompt, return_tensors="pt").to("cuda")
        attention_mask = torch.ones_like(inputs["input_ids"])
        with torch.no_grad():
            outputs = mdl.generate(
                input_ids=inputs["input_ids"],
                attention_mask=attention_mask,
                max_new_tokens=80,
                do_sample=False,    # greedy - avoids Unsloth fast-decode KV bug
                use_cache=False,    # skips broken fast_forward_inference path
                pad_token_id=tok.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        preds.append(extract_json(tok.decode(new_tokens, skip_special_tokens=True)))
    return preds

finetuned_preds = run_finetuned(model, tokenizer, test_data)
finetuned_metrics = evaluate(finetuned_preds, test_data, label="FINE-TUNED MODEL")

# ---- Build comparison lines ----
lines = []
lines.append("=" * 65)
lines.append("  CLASSIFICATION RESULTS: Original vs Fine-tuned")
lines.append("=" * 65)
lines.append(f"  {'Field':<28} {'Original':>10}  {'Fine-tuned':>10}  {'Delta':>8}")
lines.append("  " + "-" * 60)
for f in EVAL_FIELDS:
    orig  = baseline_metrics[f]["correct"] / baseline_metrics[f]["total"] * 100
    ft    = finetuned_metrics[f]["correct"] / finetuned_metrics[f]["total"] * 100
    delta = ft - orig
    sign  = "+" if delta >= 0 else ""
    lines.append(f"  {f:<28} {orig:>9.1f}%  {ft:>9.1f}%  {sign}{delta:>6.1f}%")
lines.append("=" * 65)

# ---- Failures ----
failures = [
    (i, test_data[i], finetuned_preds[i])
    for i in range(len(test_data))
    if finetuned_preds[i] != test_data[i]["output"]
]
lines.append(f"\nFailed on {len(failures)}/{len(test_data)} examples:")
for i, item, pred in failures[:10]:
    lines.append(f"\n  [{i}] Q: {item['input'][:80]}...")
    lines.append(f"       Expected : {item['output']}")
    lines.append(f"       Predicted: {pred}")

# ---- Print to console ----
for l in lines:
    print(l)

# ---- Save to txt file ----
with open("classification_results.txt", "w") as f:
    f.write("BEFORE FINE-TUNING (Original Model)\n")
    f.write("=" * 65 + "\n")
    f.write(f"  {'Field':<28} {'Accuracy':>10}  {'Correct/Total':>14}\n")
    f.write("  " + "-" * 52 + "\n")
    for fld, v in baseline_metrics.items():
        acc = v["correct"] / v["total"] * 100
        f.write(f"  {fld:<28} {acc:>9.1f}%  {v['correct']:>6}/{v['total']}\n")
    f.write("\n")
    f.write("AFTER FINE-TUNING\n")
    f.write("=" * 65 + "\n")
    f.write(f"  {'Field':<28} {'Accuracy':>10}  {'Correct/Total':>14}\n")
    f.write("  " + "-" * 52 + "\n")
    for fld, v in finetuned_metrics.items():
        acc = v["correct"] / v["total"] * 100
        f.write(f"  {fld:<28} {acc:>9.1f}%  {v['correct']:>6}/{v['total']}\n")
    f.write("\n")
    for l in lines:
        f.write(l + "\n")
    f.write("\nPER-EXAMPLE PREDICTIONS\n")
    f.write("=" * 65 + "\n")
    for i, item, pred in zip(range(len(test_data)), test_data, finetuned_preds):
        expected = item["output"]
        match = pred == expected
        f.write(f"[{i}] {'PASS' if match else 'FAIL'}\n")
        f.write(f"  Q: {item['input'][:100]}...\n")
        f.write(f"  Expected : {expected}\n")
        f.write(f"  Predicted: {pred}\n\n")

print("\nResults saved to classification_results.txt")

Fine-tuned model: 100%|██████████| 65/65 [13:36<00:00, 12.56s/it]


  FINE-TUNED MODEL
  Field                          Accuracy   Correct/Total
  ----------------------------------------------------
  questionAnswerable                84.6%      55/65
  requiresChart                     78.5%      51/65
  requiresTextOutput                86.2%      56/65
  chartType                         66.2%      43/65
  ----------------------------------------------------
  Overall accuracy                  78.8%
  Parse failures: 5/65
  CLASSIFICATION RESULTS: Original vs Fine-tuned
  Field                          Original  Fine-tuned     Delta
  ------------------------------------------------------------
  questionAnswerable                64.6%       84.6%  +  20.0%
  requiresChart                     69.2%       78.5%  +   9.2%
  requiresTextOutput                38.5%       86.2%  +  47.7%
  chartType                         64.6%       66.2%  +   1.5%

Failed on 50/65 examples:

  [0] Q: Question: How many members are currently active?

Available databa

In [25]:
import csv
# OUTPUT
output = {}
output_csv = []

for i in range(65):
    output[i+1] = {
        "input":test_data[i]['input'],
        "required-output":test_data[i]['output'],
        "baseline-output":baseline_preds[i],
        "finetune-output":finetuned_preds[i]
    }
    output_csv.append(output[i+1])

with open("jsonoutput.json",'w') as jsonfile:
  json.dump(output,jsonfile,indent=4)

fieldnames = output_csv[0].keys()

with open('csvoutput.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    dict_writer.writeheader()
    dict_writer.writerows(output_csv)



In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:23<00:00, 131.56s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:54<00:00, 57.38s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


In [ ]:
gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
if gguf_files:
    gguf_file = os.path.join("gguf_model", gguf_files[0])
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)